# Treino YOLO11 - Deteccao de EPI (Colab) - v6 (v2 limpa + imgsz 960)

Roda **sozinho**: prepara os dados, treina, e no final compara v5 x v6 no conjunto de teste real (frames do video que nunca entram no treino), salvando tudo no Drive.

**O que muda da v5:**
- **v2 limpa** (`yolo_v2_limpo.zip`): sairam as fontes que anotam uma classe so (gogglesds: so oculos; mschaberle: so capacete) e, nas que ficaram, as luvas/oculos sem caixa foram pre-anotadas pelo Grounding DINO.
- **imgsz 960** (antes 640): no video as luvas tem ~29 px e as botas ~16 px em 640; so rodar a v5 em 960 ja subiu o mAP50 real de 0.333 para 0.402.
- Base: fine-tuning a partir da v5.

**O que precisa estar no Drive** (`Meu Drive/TCC/`):
```
TCC/
  yolo_dataset/
    yolo_v2_limpo.zip
    novos_frames_dataset.zip
    construction_ppe_dataset.zip
    novos_v5.zip
  modelos/
    best_v5.pt        <- models/best.pt do PC, renomeado
```
Depois: `Ambiente de execucao > Alterar o tipo` -> **GPU T4** -> `Executar tudo`. Aba aberta, PC sem hibernar.

**Se a sessao cair**: `Executar tudo` de novo (na MESMA conta) - o treino continua de onde parou; se ja terminou, pula para a avaliacao.

**Resultados** em `Meu Drive/TCC/runs/train-colab-v6/`: `weights/best.pt`, `comparacao_real_world_test.csv/.txt`, `results.csv`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/TCC'
ZIPS = f'{DRIVE}/yolo_dataset'
RUNS = f'{DRIVE}/runs'
RUN_NAME = 'train-colab-v6'
RUN_DIR = f'{RUNS}/{RUN_NAME}'
BASE_MODEL = f'{DRIVE}/modelos/best_v5.pt'   # fine-tuning a partir da v5
V5_MODEL = BASE_MODEL
DATA_DIR = '/content/data'
CLASSES = ['helmet', 'vest', 'gloves', 'boots', 'goggles']
IMGSZ = 960

for p in [f'{ZIPS}/yolo_v2_limpo.zip', f'{ZIPS}/novos_frames_dataset.zip',
          f'{ZIPS}/construction_ppe_dataset.zip', f'{ZIPS}/novos_v5.zip', BASE_MODEL]:
    assert os.path.exists(p), f'Nao encontrado: {p}'
print('OK - todos os arquivos no Drive')

## Dados: v2 limpa + frames do video + Construction-PPE + dataset novo

In [ ]:
import shutil

for split in ['train', 'val', 'test']:
    for tipo in ['images', 'labels']:
        os.makedirs(f'{DATA_DIR}/{split}/{tipo}', exist_ok=True)


def copiar(src_root, pares):
    # pares: [(split de origem, split de destino)]
    for src, dst in pares:
        for tipo in ['images', 'labels']:
            origem = f'{src_root}/{src}/{tipo}'
            for fname in os.listdir(origem):
                shutil.copy(os.path.join(origem, fname), f'{DATA_DIR}/{dst}/{tipo}/{fname}')


!unzip -q -o "{ZIPS}/yolo_v2_limpo.zip" -d /content/v2
copiar('/content/v2/yolo_v2_limpo', [('train', 'train'), ('val', 'val'), ('test', 'test')])

# frames reais do video: train_extra (ja com oversample 8x) entra no treino;
# real_world_test fica separado e nunca e treinado
!unzip -q -o "{ZIPS}/novos_frames_dataset.zip" -d /content/novos
copiar('/content/novos/novos_frames_dataset', [('train_extra', 'train')])
REAL_TEST_DIR = '/content/real_world_test'
shutil.copytree('/content/novos/novos_frames_dataset/real_world_test', REAL_TEST_DIR, dirs_exist_ok=True)

!unzip -q -o "{ZIPS}/construction_ppe_dataset.zip" -d /content/cppe
copiar('/content/cppe/construction_ppe_raw', [('train', 'train'), ('val', 'val'), ('test', 'test')])

# dataset novo: no export do Roboflow a validacao se chama 'valid'
!unzip -q -o "{ZIPS}/novos_v5.zip" -d /content/nv5
copiar('/content/nv5/novos_v5', [('train', 'train'), ('valid', 'val'), ('test', 'test')])

In [ ]:
# Checagem antes de gastar GPU
from collections import Counter

for split in ['train', 'val', 'test']:
    imgs = os.listdir(f'{DATA_DIR}/{split}/images')
    lbls = os.listdir(f'{DATA_DIR}/{split}/labels')
    assert imgs and len(imgs) == len(lbls), f'{split}: {len(imgs)} imagens x {len(lbls)} labels'
    caixas = Counter()
    for l in lbls:
        for linha in open(f'{DATA_DIR}/{split}/labels/{l}'):
            if linha.split():
                caixas[CLASSES[int(linha.split()[0])]] += 1
    print(f'{split}: {len(imgs)} imagens | caixas: {dict(caixas)}')
print('real_world_test:', len(os.listdir(f'{REAL_TEST_DIR}/images')), 'imagens (fora do treino)')

with open(f'{DATA_DIR}/data.yaml', 'w') as f:
    f.write(f'train: {DATA_DIR}/train/images\nval: {DATA_DIR}/val/images\ntest: {DATA_DIR}/test/images\n')
    f.write(f'nc: 5\nnames: {CLASSES}\n')
with open(f'{REAL_TEST_DIR}/data.yaml', 'w') as f:
    f.write(f'train: {REAL_TEST_DIR}/images\nval: {REAL_TEST_DIR}/images\n')
    f.write(f'nc: 5\nnames: {CLASSES}\n')

## Treino (retoma sozinho se a sessao tiver caido)

imgsz 960 custa ~2x por epoca em relacao a 640; por isso 20 epocas (fine-tuning a partir da v5, que ja convergiu em 640). Resultados vao direto pro Drive.

In [ ]:
%pip install -q ultralytics
import torch
from ultralytics import YOLO

last = f'{RUN_DIR}/weights/last.pt'
terminado = False
if os.path.exists(last):
    ck = torch.load(last, map_location='cpu', weights_only=False)
    # ao terminar, o Ultralytics remove o otimizador e marca epoch = -1
    terminado = ck.get('epoch', -1) == -1
    print('treino anterior encontrado:', 'TERMINADO' if terminado else f"parou na epoca {ck.get('epoch')}")

if terminado:
    print('Pulando o treino - indo direto para a avaliacao.')
elif os.path.exists(last):
    YOLO(last).train(resume=True)
else:
    YOLO(BASE_MODEL).train(
        data=f'{DATA_DIR}/data.yaml',
        epochs=20,
        patience=8,
        batch=16,
        imgsz=IMGSZ,
        project=RUNS,
        name=RUN_NAME,
        exist_ok=True,
    )

## Veredito: v5 x v6 no conjunto de teste real

25 frames do video real, anotados a mao, que nunca entraram em nenhum treino. Avaliado em 960 (o tamanho usado no app) e em 640.

In [ ]:
import pandas as pd


def avaliar(modelo, data, nome, imgsz, split='val'):
    r = YOLO(modelo).val(data=data, imgsz=imgsz, split=split, batch=8, plots=False, verbose=False,
                         project='/content/val_tmp', name=nome, exist_ok=True)
    linhas = [{'modelo': nome, 'imgsz': imgsz, 'classe': 'GERAL', 'mAP50': r.box.map50, 'mAP50-95': r.box.map,
               'precisao': r.box.mp, 'recall': r.box.mr}]
    for i, c in enumerate(r.box.ap_class_index):
        linhas.append({'modelo': nome, 'imgsz': imgsz, 'classe': CLASSES[int(c)], 'mAP50': r.box.ap50[i],
                       'mAP50-95': r.box.ap[i], 'precisao': r.box.p[i], 'recall': r.box.r[i]})
    return linhas


V6_MODEL = f'{RUN_DIR}/weights/best.pt'
linhas = []
for sz in [960, 640]:
    linhas += avaliar(V5_MODEL, f'{REAL_TEST_DIR}/data.yaml', 'v5', sz)
    linhas += avaliar(V6_MODEL, f'{REAL_TEST_DIR}/data.yaml', 'v6', sz)
res = pd.DataFrame(linhas).round(3)
texto = ''
for sz in [960, 640]:
    t = res[res.imgsz == sz].pivot(index='classe', columns='modelo', values=['mAP50', 'recall', 'precisao'])
    texto += f'=== CONJUNTO DE TESTE REAL (video) - imgsz {sz} ===\n' + t.to_string() + '\n\n'
print(texto)

teste = pd.DataFrame(avaliar(V6_MODEL, f'{DATA_DIR}/data.yaml', 'v6-split-teste', IMGSZ, split='test')).round(3)
print('=== v6 no split de teste (otimista, so para referencia) ===')
print(teste.to_string(index=False))

res.to_csv(f'{RUN_DIR}/comparacao_real_world_test.csv', index=False)
with open(f'{RUN_DIR}/comparacao_real_world_test.txt', 'w') as f:
    f.write(texto + '=== v6 no split de teste ===\n' + teste.to_string(index=False) + '\n')
print(f'\nSalvo em {RUN_DIR}/comparacao_real_world_test.csv e .txt')